# Notebook 3 — Evaluation

Compares base Llama 3.2 vs fine-tuned model on the test set.

**Metrics:** JSON validity, least-privilege compliance, NIST mapping accuracy, ROUGE-L

In [ ]:
import sys, json
sys.path.append('../src')
from data_utils import load_jsonl, extract_json_from_output, extract_actions, is_valid_policy
from inference import load_model, generate_policy

## 1. Load test set

In [ ]:
test_data = load_jsonl('../data/processed/test.jsonl')
print(f'{len(test_data)} test examples')

## 2. Run inference — base model and fine-tuned

In [ ]:
# Load fine-tuned model
ft_model, ft_tokenizer = load_model('../iam-policy-adapter')

ft_outputs = [generate_policy(ex['input'], ft_model, ft_tokenizer) for ex in test_data]
print('Fine-tuned inference done')

## 3. Metric 1 — JSON validity

In [ ]:
valid = [o for o in ft_outputs if not o.get('parse_error')]
print(f'JSON validity: {len(valid)}/{len(ft_outputs)} = {len(valid)/len(ft_outputs):.1%}')

## 4. Metric 2 — Least-privilege check

In [ ]:
def check_least_privilege(output, ground_truth):
    if output.get('parse_error'):
        return False
    try:
        gt_actions = extract_actions(ground_truth['output']['policy'])
        pred_actions = extract_actions(output['policy'])
        over_provisioned = pred_actions - gt_actions
        return len(over_provisioned) == 0
    except Exception:
        return False

lp_results = [check_least_privilege(o, gt) for o, gt in zip(ft_outputs, test_data)]
print(f'Least-privilege compliance: {sum(lp_results)}/{len(lp_results)} = {sum(lp_results)/len(lp_results):.1%}')

## 5. Metric 3 — ROUGE-L

In [ ]:
from evaluate import load as load_metric
rouge = load_metric('rouge')

predictions = [json.dumps(o) for o in ft_outputs]
references = [json.dumps(ex['output']) for ex in test_data]
results = rouge.compute(predictions=predictions, references=references)
print(f"ROUGE-L: {results['rougeL']:.3f}")

## 6. Results summary

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'Metric': ['JSON validity', 'Least-privilege', 'ROUGE-L'],
    'Base Llama 3.2': ['—', '—', '—'],
    'Fine-tuned': [
        f"{len(valid)/len(ft_outputs):.1%}",
        f"{sum(lp_results)/len(lp_results):.1%}",
        f"{results['rougeL']:.3f}"
    ]
})
summary